Konfiguracja i parametry

In [17]:
# %% [1] Konfiguracja i Parametry
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# =========================================================
# 1. Ścieżki do plików (I/O)
# =========================================================
PATH_TIME_PULS  = r"C:\DanePYTHON\Puls (moje)\Input_data\Time Puls dp 15m.csv"
PATH_PROGRAMMES = r"C:\DanePYTHON\Puls (moje)\Input_data\Programmes Puls pg.csv"
PATH_EVENTS     = r"W:\Dzial Business process\Dzial Analiz\OSOBISTE\RAFAŁ\Wewnętrzne\Events_ratings.xlsx"
PATH_PREMIERY   = r"C:\DanePYTHON\Puls (moje)\Input_data\Premiery.xlsx"
PATH_ATV        = r"C:\DanePYTHON\Puls (moje)\Input_data\Time ATV_hourly_TSV.csv"
PATH_PRED_NEWDATA = r"C:\DanePYTHON\Puls (moje)\Input_data\df_pred NEWDATA.xlsx" 

PATH_OUT_DATA   = r"C:\DanePYTHON\Puls (moje)\Processed data\data.xlsx"
PATH_OUT_PRED   = r"C:\DanePYTHON\Puls (moje)\Processed data\df_pred.xlsx"
PATH_OUT_FORECAST = r"C:\DanePYTHON\Puls (moje)\Output_data\Forecasted_SHR.xlsx"

# =========================================================
# 2. Parametry biznesowe i inżynieria cech
# =========================================================
CHANNEL_TARGET = "PULS"
DEPENDENT_VAR = "SHR"

# NOWE ZMIENNE DYNAMICZNE:
# Lagi do wyliczenia średniej 'SHR_lag_mean' (1, 2, 3 i 4 tygodnie wstecz)
LAGS_FOR_MEAN = (672, 1344, 2016, 2688)
# Roczny lag
LAG_YEAR = 35040
# Okna dla średnich ruchomych (4 tygodnie i 1 rok)
WINDOWS = (2688, 35040)

# Zaktualizowana lista predyktorów o nowe zmienne
PREDICTORS = [
    "Hour", "Hour_sin", "Hour_cos", 
    "Weekday", "Weekday_sin", "Weekday_cos",
    "Week", "Week_sin", "Week_cos", 
    "Month", "Month_sin", "Month_cos",
    "SHR_lag_mean", "SHR_lag_35040", 
    "SHR_rolling_mean_2688", "SHR_rolling_mean_35040",
    "Duration", "Programme", "Type", "Label", "Rank", "Premiere", "Sezon", "ATV", "Holiday"
]

CATEGORICAL_COLS = ["Hour", "Weekday", "Week", "Month", "Programme", "Type", "Label", "Sezon", "Holiday"]

# =========================================================
# 3. Parametry Modelu i Walidacji
# =========================================================
VALIDATION_HORIZON_DAYS = 45
OPTUNA_TRIALS = 20
EARLY_STOPPING_ROUNDS = 50
NEXT_MONTH_END = "2026-09-30 23:45:00"

# =========================================================
# 4. Słowniki i mapowania
# =========================================================
POLISH_CHARS = str.maketrans("ąćęłńóśźżĄĆĘŁŃÓŚŹŻ", "acelnoszzACELNOSZZ")

TYPE_MAPPING = {
    "Film/serial animowany dla dzieci": "Film",
    "Film/serial dokumentalny - podrozniczy - przyrodniczy": "Film",
    "Filmy fabularne": "Film",
    "Inne programy rozrywkowe": "Program",
    "Kabaret": "Kabaret",
    "Magazyny naukowe/edukacyjne/popularno naukowe": "Magazyn",
    "Magazyny reportazy/reportaze": "Magazyn",
    "Programy poradnikowe": "Program",
    "Seriale": "Serial",
    "Telenowele fabularne": "Serial",
    "Teleturnieje": "Program",
    "Wiadomosci": "Wiadomosci"
}
EXCEPTIONS_SERIAL = ["na jedwabnym szlaku", "dyzur", "z archiwum policji"]

TIME_START_LN = pd.to_timedelta("02:00:00")
TIME_END_LN = pd.to_timedelta("06:00:00")
TIME_START_LOMBARD = pd.to_timedelta("19:00:00")
TIME_END_LOMBARD = pd.to_timedelta("20:00:00")

Funkcje pomocnicze

In [18]:
# %% [2] Funkcje pomocnicze
def parse_float(series):
    """Zamienia stringi z przecinkiem na wartości numeryczne."""
    return pd.to_numeric(series.astype(str).str.replace(",", "."), errors="coerce")

def fix_over_24h_time(df, date_col, time_col):
    """Koryguje daty i godziny zapisane powyżej 24:00:00 (np. 25:59)."""
    df[date_col] = pd.to_datetime(df[date_col], format="%Y-%m-%d")
    df[time_col] = pd.to_timedelta(df[time_col])
    
    days_to_add = df[time_col].dt.days
    df[date_col] = df[date_col] + pd.to_timedelta(days_to_add, unit="d")
    df[time_col] = df[time_col] - pd.to_timedelta(days_to_add, unit="d")
    return df

Import plików i pre-processing bazy

In [19]:
# %% [3] Import plików i pre-processing (DP & PG)
# --- BAZA (DP) ---
df_dp = pd.read_csv(PATH_TIME_PULS, skiprows=10, header=1, sep=",", quotechar='"', usecols=range(6))
df_dp.columns = ["Channel", "Date", "Time", "AMR", "AMR%", "SHR"]

df_dp = fix_over_24h_time(df_dp, "Date", "Time")
df_dp["DT"] = df_dp["Date"] + df_dp["Time"]

for col in ["AMR", "AMR%", "SHR"]:
    df_dp[col] = parse_float(df_dp[col])

# --- PROGRAMY (PG) ---
df_pg = pd.read_csv(PATH_PROGRAMMES, skiprows=10, header=1, sep=",", quotechar='"', usecols=range(9))
df_pg.columns = ["Channel", "Date", "Time", "Time_end", "Duration", "Programme", "Type", "AMR%", "SHR"]

df_pg = fix_over_24h_time(df_pg, "Date", "Time")
df_pg["Time_end"] = pd.to_timedelta(df_pg["Time_end"])
df_pg["Time_end"] = df_pg["Time_end"] - pd.to_timedelta(df_pg["Time_end"].dt.days, unit="d")

df_pg["Duration"] = pd.to_timedelta(df_pg["Duration"])
df_pg["Start_DT"] = df_pg["Date"] + df_pg["Time"]
df_pg["End_DT"]   = df_pg["Start_DT"] + df_pg["Duration"]

df_pg["AMR%"] = parse_float(df_pg["AMR%"])
df_pg["SHR"]  = parse_float(df_pg["SHR"])

# --- MERGE (DP + PG) ---
df_pg_filtered = df_pg[df_pg["Duration"] >= pd.Timedelta(minutes=15)].sort_values(["Channel", "Start_DT"])
df = df_dp.sort_values(["Channel", "DT"]).copy()

df_pg_sub = df_pg_filtered[["Channel", "Start_DT", "End_DT", "Duration", "Programme", "Type", "AMR%", "SHR"]].rename(
    columns={"AMR%": "AMR%_pg", "SHR": "SHR_pg"}
)

df = pd.merge_asof(
    df, df_pg_sub, left_on="DT", right_on="Start_DT", by="Channel", direction="backward"
)

# Walidacja czasu trwania programu
valid_match = df["DT"] < df["End_DT"]
cols_from_pg = ["Duration", "Programme", "Type", "AMR%_pg", "SHR_pg"]
df.loc[~valid_match, cols_from_pg] = pd.NA

# Standaryzacja nazw programów i mapowanie typów
df["Programme"] = (
    df["Programme"].fillna("break")
    .str.translate(POLISH_CHARS).str.lower()
    .str.replace(r'[^\w\s]', '', regex=True)
    .str.replace(r'\s+', ' ', regex=True).str.strip()
)

df["Type"] = df["Type"].fillna("Break").replace(TYPE_MAPPING)
df.loc[df["Programme"].isin(EXCEPTIONS_SERIAL), "Type"] = "Serial"
df.loc[df["Type"] == "Film", "Programme"] = "film"

mask_ln = (df["Time"] >= TIME_START_LN) & (df["Time"] < TIME_END_LN)
df.loc[mask_ln, "Programme"] = "ln"

df["AMR%_pg"] = pd.to_numeric(df["AMR%_pg"], errors="coerce").fillna(df["AMR%"])
df["SHR_pg"]  = pd.to_numeric(df["SHR_pg"], errors="coerce").fillna(df["SHR"])

Inżynieria cech

In [20]:
# %% [4] Feature Engineering
# 1. Kodowanie trygonometryczne czasu
time_components = {
    "Hour": (df["DT"].dt.hour, 24),
    "Weekday": (df["DT"].dt.weekday + 1, 7),
    "Week": (df["DT"].dt.isocalendar().week.astype(int), 52),
    "Month": (df["DT"].dt.month, 12)
}

for name, (series, period) in time_components.items():
    df[name] = series.astype("category")
    df[f"{name}_sin"] = np.sin(2 * np.pi * series / period)
    df[f"{name}_cos"] = np.cos(2 * np.pi * series / period)

# 2. Events z excela
df_events = pd.read_excel(PATH_EVENTS, sheet_name="Events_full")
df_events["DT"] = pd.to_datetime(df_events["DT"])
df_events["Event_End_DT"] = df_events["DT"] + pd.to_timedelta(df_events["Duration"].astype(str))
df_events = df_events[["DT", "Event_End_DT", "Label", "Rank"]].sort_values("DT")

df = pd.merge_asof(df.sort_values("DT"), df_events, on="DT", direction="backward")
df.loc[df["DT"] >= df["Event_End_DT"], ["Label", "Rank"]] = pd.NA
df = df.drop(columns=["Event_End_DT"]).sort_values(["Channel", "DT"])

df["Label"] = df["Label"].fillna("Brak")
df["Rank"]  = pd.to_numeric(df["Rank"]).fillna(0)

# 2.5. Holidays z excela
df_holidays = pd.read_excel(PATH_EVENTS, sheet_name="Holiday_full").rename(columns={"Data": "Date_join"})
df_holidays["Date_join"] = pd.to_datetime(df_holidays["Date_join"])
df["Date_join"] = df["DT"].dt.normalize()
df = df.merge(df_holidays[["Date_join", "Holiday"]], on="Date_join", how="left")
df["Holiday"] = df["Holiday"].fillna("Brak")
df = df.drop(columns=["Date_join"]).sort_values(["Channel", "DT"])

# 3. ATV
df_atv = pd.read_csv(PATH_ATV, skiprows=10, sep=",", quotechar='"', usecols=[0, 1, 2])
df_atv.columns = ["Date", "Time", "ATV"]
df_atv = fix_over_24h_time(df_atv, "Date", "Time")
df_atv["ATV"] = parse_float(df_atv["ATV"])
df_atv["DT_hour"] = df_atv["Date"] + df_atv["Time"]

df["DT_hour"] = df["DT"].dt.floor("h")
df = df.merge(df_atv[["DT_hour", "ATV"]], on="DT_hour", how="left").drop(columns=["DT_hour"])

# 4. Premiery (Lombard)
if pd.api.types.is_timedelta64_dtype(df["Duration"]):
    df["Duration"] = df["Duration"].dt.total_seconds() / 60.0
df["Duration"] = df["Duration"].fillna(12.0)

df_premiery = pd.read_excel(PATH_PREMIERY)
df_premiery["Od"] = pd.to_datetime(df_premiery["Od"])
df_premiery["Do"] = pd.to_datetime(df_premiery["Do"])

df["Premiere"] = 0
df["Sezon"] = "Brak"

mask_lombard = (df["Programme"] == "lombard zycie pod zastaw") & (df["Weekday"].isin([1, 2, 3, 4, 5])) & (df["Time"] >= TIME_START_LOMBARD) & (df["Time"] < TIME_END_LOMBARD)

for _, row in df_premiery.iterrows():
    mask_date = (df["DT"] >= row["Od"]) & (df["DT"] <= (row["Do"] + pd.Timedelta(days=1)))
    mask_final = mask_lombard & mask_date
    df.loc[mask_final, "Premiere"] = 1
    df.loc[mask_final, "Sezon"] = row["Sezon"]

Generowanie zmiennych dynamicznych

In [21]:
# %% [5] Dynamics Variables - Lagi i Średnie ruchome
df = df.sort_values(by=["Channel", "DT"]).reset_index(drop=True)

# 1. SHR_lag_mean (Średnia z opóźnień 672, 1344, 2016, 2688)
temp_lag_cols = []
for lag in LAGS_FOR_MEAN:
    col_name = f"temp_lag_{lag}"
    df[col_name] = df.groupby("Channel")["SHR"].shift(lag)
    temp_lag_cols.append(col_name)

# Obliczenie średniej na osi wierszy z pominięciem braków danych (NaN)
df["SHR_lag_mean"] = df[temp_lag_cols].mean(axis=1)
# Usunięcie tymczasowych kolumn
df = df.drop(columns=temp_lag_cols)

# 2. SHR_lag_35040 (Roczne opóźnienie)
df["SHR_lag_35040"] = df.groupby("Channel")["SHR"].shift(LAG_YEAR)

# 3. Rolling Means (Dla okien 2688 i 35040)
for window in WINDOWS:
    df[f"SHR_rolling_mean_{window}"] = df.groupby("Channel")["SHR"].transform(
        lambda x: x.rolling(window=window, min_periods=1).mean()
    )

# Finalna tabela do modelowania
data = df[["DT"] + PREDICTORS + [DEPENDENT_VAR]].copy()
data.to_excel(PATH_OUT_DATA, index=False)

Budowa i trening modelu XGBoost

In [22]:
# %% [6] XGBoost Model i Optymalizacja Bayesowska
import xgboost as xgb
import optuna
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Konwersja typów kategorycznych
for col in CATEGORICAL_COLS:
    if col in data.columns:
        data[col] = data[col].astype(str).astype("category")

# Podział OOT (Out-of-Time Validation)
split_date = data["DT"].max() - pd.Timedelta(days=VALIDATION_HORIZON_DAYS)
data = data.sort_values("DT")

train_data = data[data["DT"] < split_date].copy()
valid_data = data[data["DT"] >= split_date].copy()

X_train, y_train = train_data[PREDICTORS], train_data[DEPENDENT_VAR]
X_valid, y_valid = valid_data[PREDICTORS], valid_data[DEPENDENT_VAR]

print(f"Trening: {train_data['DT'].min()} do {train_data['DT'].max()} ({len(train_data)} rzędów)")
print(f"Walidacja: {valid_data['DT'].min()} do {valid_data['DT'].max()} ({len(valid_data)} rzędów)")

def objective(trial):
    param = {
        "objective": "reg:squarederror",
        "tree_method": "hist",
        "enable_categorical": True,
        "random_state": 42,
        "n_estimators": trial.suggest_int("n_estimators", 200, 1500),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "max_depth": trial.suggest_int("max_depth", 4, 12),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma": trial.suggest_float("gamma", 1e-8, 1.0, log=True),
        "early_stopping_rounds": EARLY_STOPPING_ROUNDS
    }
    
    model = xgb.XGBRegressor(**param)
    model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)
    preds = model.predict(X_valid)
    return np.sqrt(mean_squared_error(y_valid, preds))

print("\nOptymalizacja hiperparametrów...")
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=OPTUNA_TRIALS) 

print("\nTrenowanie finalnego modelu...")
final_params = study.best_params
final_params.update({
    "objective": "reg:squarederror", "tree_method": "hist",
    "enable_categorical": True, "random_state": 42, "early_stopping_rounds": EARLY_STOPPING_ROUNDS
})

final_model = xgb.XGBRegressor(**final_params)
final_model.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_valid, y_valid)], verbose=100)

final_preds = final_model.predict(X_valid)
print(f"\nFinalne Metryki OOT:\nRMSE: {np.sqrt(mean_squared_error(y_valid, final_preds)):.4f}\nMAE:  {mean_absolute_error(y_valid, final_preds):.4f}")

[I 2026-08-26 14:25:48,467] A new study created in memory with name: no-name-95affeed-cc46-41a2-95ce-3aa9bd3febd5


Trening: 2024-01-01 02:15:00 do 2026-07-11 01:30:00 (88606 rzędów)
Walidacja: 2026-07-11 01:45:00 do 2026-08-25 01:45:00 (4321 rzędów)

Optymalizacja hiperparametrów...


[I 2026-08-26 14:25:55,859] Trial 0 finished with value: 2.020752319006923 and parameters: {'n_estimators': 1308, 'learning_rate': 0.012251787642994642, 'max_depth': 4, 'min_child_weight': 8, 'subsample': 0.82030492213116, 'colsample_bytree': 0.803643725895316, 'gamma': 0.21531017566149857}. Best is trial 0 with value: 2.020752319006923.
[I 2026-08-26 14:26:01,918] Trial 1 finished with value: 2.066901545055084 and parameters: {'n_estimators': 856, 'learning_rate': 0.17557276340969027, 'max_depth': 11, 'min_child_weight': 14, 'subsample': 0.7133828180634094, 'colsample_bytree': 0.729408316371606, 'gamma': 0.0002401253128544854}. Best is trial 0 with value: 2.020752319006923.
[I 2026-08-26 14:26:09,072] Trial 2 finished with value: 2.02627689738751 and parameters: {'n_estimators': 311, 'learning_rate': 0.011062245548863814, 'max_depth': 9, 'min_child_weight': 11, 'subsample': 0.7924320790404499, 'colsample_bytree': 0.7255802360429068, 'gamma': 6.061541499419522e-05}. Best is trial 0 wit


Trenowanie finalnego modelu...
[0]	validation_0-rmse:1.99667	validation_1-rmse:2.34089
[100]	validation_0-rmse:1.62373	validation_1-rmse:2.01215
[122]	validation_0-rmse:1.60249	validation_1-rmse:2.01907

Finalne Metryki OOT:
RMSE: 2.0037
MAE:  1.3395


Generowanie szkieletu predykcyjnego

In [23]:
# %% [7] Szkielet predykcyjny (df_pred)
future_dates = pd.date_range(
    start=data["DT"].max() + pd.Timedelta(minutes=15),
    end=NEXT_MONTH_END,
    freq="15min"
)
df_pred = pd.DataFrame({"DT": future_dates})

time_components_pred = {
    "Hour": (df_pred["DT"].dt.hour, 24),
    "Weekday": (df_pred["DT"].dt.weekday + 1, 7),
    "Week": (df_pred["DT"].dt.isocalendar().week.astype(int), 52),
    "Month": (df_pred["DT"].dt.month, 12)
}

for name, (series, period) in time_components_pred.items():
    df_pred[f"{name}_sin"] = np.sin(2 * np.pi * series / period)
    df_pred[f"{name}_cos"] = np.cos(2 * np.pi * series / period)
    df_pred[name] = series.astype(str).astype("category")

df_pred["Programme"], df_pred["Type"] = pd.NA, pd.NA
df_pred["Duration"], df_pred["Premiere"], df_pred["Sezon"] = np.nan, np.nan, pd.NA
df_pred["ATV"] = np.nan

# Zaktualizowana inicjalizacja pustych zmiennych dynamicznych
df_pred["SHR_lag_mean"] = np.nan
df_pred["SHR_lag_35040"] = np.nan

for window in WINDOWS: 
    df_pred[f"SHR_rolling_mean_{window}"] = np.nan

# Events & Holidays dla predykcji
df_events = pd.read_excel(PATH_EVENTS, sheet_name="Events_full")
df_events["DT"] = pd.to_datetime(df_events["DT"])
df_events["Event_End_DT"] = df_events["DT"] + pd.to_timedelta(df_events["Duration"].astype(str))
df_events = df_events[["DT", "Event_End_DT", "Label", "Rank"]].sort_values("DT")

df_pred = pd.merge_asof(df_pred.sort_values("DT"), df_events, on="DT", direction="backward")
df_pred.loc[df_pred["DT"] >= df_pred["Event_End_DT"], ["Label", "Rank"]] = pd.NA
df_pred = df_pred.drop(columns=["Event_End_DT"]).sort_values("DT")
df_pred["Label"] = df_pred["Label"].fillna("Brak").astype(str).astype("category")
df_pred["Rank"]  = pd.to_numeric(df_pred["Rank"]).fillna(0)

df_holidays = pd.read_excel(PATH_EVENTS, sheet_name="Holiday_full").rename(columns={"Data": "Date_join"})
df_holidays["Date_join"] = pd.to_datetime(df_holidays["Date_join"])
df_pred["Date_join"] = df_pred["DT"].dt.normalize()
df_pred = df_pred.merge(df_holidays[["Date_join", "Holiday"]], on="Date_join", how="left")
df_pred["Holiday"] = df_pred["Holiday"].fillna("Brak").astype(str).astype("category")
df_pred = df_pred.drop(columns=["Date_join"]).sort_values("DT")

print(f"Struktura df_pred gotowa. Kształt: {df_pred.shape}")

# Zapis pliku wynikowego z użyciem ścieżki względnej
df_pred.to_excel("Processed data/df_pred.xlsx", index=False)

Struktura df_pred gotowa. Kształt: (3544, 26)


Autoregresyjna prognoza iteracyjna

In [24]:
# %% [8] Prognoza iteracyjna
df_newdata = pd.read_excel(PATH_PRED_NEWDATA)
df_newdata["DT"] = pd.to_datetime(df_newdata["DT"])
df_newdata = df_newdata.sort_values("DT").reset_index(drop=True)

df_premiery = pd.read_excel(PATH_PREMIERY)
df_premiery["Od"] = pd.to_datetime(df_premiery["Od"])
df_premiery["Do"] = pd.to_datetime(df_premiery["Do"])

# Uzupełnienie Premiere i Sezon
df_newdata["Time"] = pd.to_timedelta(df_newdata["DT"].dt.strftime('%H:%M:%S'))
df_newdata["Premiere"], df_newdata["Sezon"] = 0, "Brak"

mask_lombard_new = (
    (df_newdata["Programme"] == "lombard zycie pod zastaw") & 
    (df_newdata["Weekday"].astype(str).isin(["1", "2", "3", "4", "5"])) & 
    (df_newdata["Time"] >= TIME_START_LOMBARD) & 
    (df_newdata["Time"] < TIME_END_LOMBARD)
)

for _, row in df_premiery.iterrows():
    mask_date = (df_newdata["DT"] >= row["Od"]) & (df_newdata["DT"] <= (row["Do"] + pd.Timedelta(days=1)))
    mask_final = mask_lombard_new & mask_date
    df_newdata.loc[mask_final, "Premiere"] = 1
    df_newdata.loc[mask_final, "Sezon"] = row["Sezon"]
df_newdata = df_newdata.drop(columns=["Time"])

for col in CATEGORICAL_COLS:
    if col in df_newdata.columns:
        df_newdata[col] = df_newdata[col].astype(str).astype("category")

# Zabezpieczenie rozmiaru bufora (Musi pomieścić roczny lag i roczną średnią ruchomą)
max_history_needed = max(max(LAGS_FOR_MEAN), LAG_YEAR, max(WINDOWS))

historical_shr = data[DEPENDENT_VAR].tolist() 
# Upewnienie się, że mamy wystarczająco dużo danych w buforze
if len(historical_shr) < max_history_needed:
    raise ValueError(f"Zbyt krótka historia! Wymagane {max_history_needed} kwadransów, dostępne: {len(historical_shr)}.")

buffer_shr = historical_shr[-max_history_needed:]

print("Rozpoczynam iteracyjną prognozę. Może to potrwać dłuższą chwilę...")
predicted_shr = []

for i in range(len(df_newdata)):
    
    # 1. SHR_lag_mean (wyliczenie średniej punktowej z bufora)
    lag_values = [buffer_shr[-lag] for lag in LAGS_FOR_MEAN]
    df_newdata.at[i, "SHR_lag_mean"] = np.mean(lag_values)
    
    # 2. SHR_lag_35040
    df_newdata.at[i, "SHR_lag_35040"] = buffer_shr[-LAG_YEAR]
        
    # 3. SHR_rolling_means
    for window in WINDOWS:
        df_newdata.at[i, f"SHR_rolling_mean_{window}"] = np.mean(buffer_shr[-window:])
        
    # Predykcja
    row_df = df_newdata.loc[[i], PREDICTORS]
    pred = final_model.predict(row_df)[0]
    pred = max(0.0, pred)
    
    predicted_shr.append(pred)
    buffer_shr.append(pred)
    buffer_shr.pop(0) 
    
    if (i + 1) % 500 == 0:
        print(f" Przetworzono {i + 1} z {len(df_newdata)} kwadransów...")

df_newdata["SHR"] = predicted_shr
print("Prognoza zakończona sukcesem!")
df_newdata.to_excel(PATH_OUT_FORECAST, index=False)

Rozpoczynam iteracyjną prognozę. Może to potrwać dłuższą chwilę...
 Przetworzono 500 z 3544 kwadransów...
 Przetworzono 1000 z 3544 kwadransów...
 Przetworzono 1500 z 3544 kwadransów...
 Przetworzono 2000 z 3544 kwadransów...
 Przetworzono 2500 z 3544 kwadransów...
 Przetworzono 3000 z 3544 kwadransów...
 Przetworzono 3500 z 3544 kwadransów...
Prognoza zakończona sukcesem!
